# v2 Self-Improving Ensemble Prediction Engine
Run these cells using the **Google Colab Extension** kernel.

## 1. Install Dependencies + Sync Code

In [26]:
%pip install xgboost scikit-learn pandas numpy networkx catboost -q

import os
%cd /content

if not os.path.exists('/content/ultimate_predictor'):
    !git clone https://github.com/sagarsinde/ultimate_predictor.git
    %cd /content/ultimate_predictor/
else:
    %cd /content/ultimate_predictor/
    !git pull origin main

/content
/content/ultimate_predictor
remote: Enumerating objects: 11, done.
remote: Counting objects: 100% (11/11), done.
remote: Compressing objects: 100% (3/3), done.
remote: Total 8 (delta 5), reused 8 (delta 5), pack-reused 0 (from 0)
Unpacking objects: 100% (8/8), 1.53 KiB | 1.53 MiB/s, done.
From https://github.com/sagarsinde/ultimate_predictor
 * branch            main       -> FETCH_HEAD
   449de67..c192911  main       -> origin/main
Updating 449de67..c192911
Fast-forward
 v2/update_weights.py | 53 ++++++++++++++++++++++++++++++++++++++++++++++++++++
 v2/validator.py      | 22 +++++++++++-----------
 2 files changed, 64 insertions(+), 11 deletions(-)
 create mode 100644 v2/update_weights.py


In [ ]:
%cd /content/ultimate_predictor
!python v2/update_weights.py


In [25]:
%cd /content/ultimate_predictor
!python v2/evaluate_june.py


/content/ultimate_predictor

  EVALUATING: KALYAN (2026-06) — ONLY full_freq (Morning, Evening, Jodi)
  Date         | Actual M | Pred M (Top3)        | Actual E | Pred E (Top3)        | Actual Jodi  | Pred Jodi (Top4)                   
  -------------+----------+----------------------+----------+----------------------+--------------+------------------------------------
  2026-06-01   | 0        | 1, 7, 5 ❌            | 2        | 3, 1, 4 ❌            | 02           | 13, 73, 53, 11 ❌                   
  2026-06-02   | 3        | 1, 7, 5 ❌            | 8        | 3, 1, 4 ❌            | 38           | 73, 13, 53, 71 ❌                   
  2026-06-03   | 3        | 1, 7, 5 ❌            | 1        | 3, 1, 4 ✅            | 31           | 73, 13, 53, 71 ❌                   
  2026-06-04   | 8        | 1, 7, 5 ❌            | 3        | 3, 1, 4 ⭐            | 83           | 73, 13, 53, 71 ❌                   
  2026-06-05   | 8        | 1, 7, 5 ❌            | 0        | 3, 1, 4 ❌           

In [27]:
%cd /content/ultimate_predictor
!python v2/update_weights.py


/content/ultimate_predictor
Error: No avg_metrics in state file.
Error: No avg_metrics in state file.


## 2. FIRST TIME SETUP: Run Full Backtest (Feature Ablation + Model Pruning)
Run this ONCE. It will:
1. Test every feature group and remove useless ones
2. Run walk-forward validation on 5 monthly periods
3. Learn model weights from Brier Scores
4. Prune weak models
5. Build confidence calibration

This takes ~5-10 minutes on Colab GPU.

In [13]:
%cd /content/ultimate_predictor/
!python -m v2.run_backtest kalyan

/content/ultimate_predictor

######################################################################
  v2 SELF-IMPROVING ENSEMBLE — FULL BACKTEST: KALYAN
######################################################################

[STEP 1/5] Feature Ablation...

  FEATURE ABLATION: KALYAN

  Running baseline with ALL feature groups...
  Baseline avg Brier: 0.0956

  Testing WITHOUT 'lags'... Brier=0.0956 (Δ=-0.0000) → REMOVE ❌
  Testing WITHOUT 'days_since_hit'... Brier=0.0961 (Δ=+0.0005) → KEEP ✅
  Testing WITHOUT 'hits_last_7'... Brier=0.0957 (Δ=+0.0001) → KEEP ✅
  Testing WITHOUT 'day_of_week'... Brier=0.0957 (Δ=+0.0001) → KEEP ✅
  Testing WITHOUT 'morning_evening_corr'... Brier=0.0959 (Δ=+0.0003) → KEEP ✅
  Testing WITHOUT 'hot_cold_ratio'... Brier=0.0957 (Δ=+0.0000) → KEEP ✅
  Testing WITHOUT 'gap_velocity'... Brier=0.0956 (Δ=-0.0000) → REMOVE ❌

  Surviving features: ['days_since_hit', 'hits_last_7', 'day_of_week', 'morning_evening_corr', 'hot_cold_ratio']
  Removed features: ['lags', 

In [7]:
%cd /content/ultimate_predictor/
!python -m v2.run_backtest mb

/content/ultimate_predictor

######################################################################
  v2 SELF-IMPROVING ENSEMBLE — FULL BACKTEST: MB
######################################################################

[STEP 1/5] Feature Ablation...

  FEATURE ABLATION: MB

  Running baseline with ALL feature groups...
  Baseline avg Brier: 0.0967

  Testing WITHOUT 'lags'... Brier=0.0969 (Δ=+0.0002) → KEEP ✅
  Testing WITHOUT 'days_since_hit'... Brier=0.0968 (Δ=+0.0001) → KEEP ✅
  Testing WITHOUT 'day_of_week'... Brier=0.0967 (Δ=-0.0000) → REMOVE ❌
  Testing WITHOUT 'morning_evening_corr'... Brier=0.0967 (Δ=+0.0000) → KEEP ✅
  Testing WITHOUT 'hot_cold_ratio'... Brier=0.0969 (Δ=+0.0002) → KEEP ✅

  Model            Top1-M  Top1-E  Top3-M  Top3-E  Brier-M  Brier-E Periods
  --------------- ------- ------- ------- ------- -------- -------- -------
  1m_freq           5.7%  11.4%  25.7%  28.6%   0.0930   0.0916       5
  1m_markov         8.6%  20.0%  40.0%  40.0%   0.0903   0.0896    

## 3. DAILY USE: Predict Tomorrow
Run these cells after updating your CSVs and pushing to GitHub.
Uses the learned weights and surviving models from the backtest.

In [14]:
%cd /content/ultimate_predictor/
!python -m v2.run_predict kalyan

/content/ultimate_predictor

######################################################################
  v2 SELF-IMPROVING ENSEMBLE — PREDICTION: KALYAN
######################################################################

  Loading state: 12 surviving models, 5 feature groups

══════════════════════════════════════════════════════════════════════
  KALYAN PREDICTION: 2026-07-03 (Friday)
══════════════════════════════════════════════════════════════════════

  MORNING (Open):  [❌ SKIP]
  ────────────────────────────────────────────────────────────
  Rank   Digit   Ensemble Prob   Hist. Hit Rate  
  ────── ─────── ─────────────── ────────────────
  1      0       16.4%           —               
  2      9       15.0%           —               
  3      3       12.6%           —               
  4      1       9.8%            —               
  5      5       9.2%            —               

  EVENING (Close):  [❌ SKIP]
  ────────────────────────────────────────────────────────────
  Ra

In [8]:
%cd /content/ultimate_predictor/
!python -m v2.run_predict mb

/content/ultimate_predictor

######################################################################
  v2 SELF-IMPROVING ENSEMBLE — PREDICTION: MB
######################################################################

  Loading state: 16 surviving models, 4 feature groups

══════════════════════════════════════════════════════════════════════
  MB PREDICTION: 2026-07-01 (Wednesday)
══════════════════════════════════════════════════════════════════════

  MORNING (Open):  [❌ SKIP]
  ────────────────────────────────────────────────────────────
  Rank   Digit   Ensemble Prob   Hist. Hit Rate  
  ────── ─────── ─────────────── ────────────────
  1      3       33.1%           —               
  2      6       9.8%            —               
  3      2       8.8%            —               
  4      1       7.8%            —               
  5      4       7.8%            —               

  EVENING (Close):  [⚠️ MARGINAL]
  ────────────────────────────────────────────────────────────
  Ra

## 4. V3 ENGINE (CatBoost & DowFreq)
Run the new v3 engine tests here!

In [ ]:
%cd /content/ultimate_predictor/
!python -m v3.run_backtest kalyan

In [ ]:
%cd /content/ultimate_predictor/
!python -m v3.run_backtest mb

In [ ]:
%cd /content/ultimate_predictor/
!python -m v3.run_predict kalyan

In [ ]:
%cd /content/ultimate_predictor/
!python -m v3.run_predict mb